In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
import warnings
import xgboost as xgb
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme()

In [2]:
train = pd.read_csv("data/track_k_ecommerce_returns_train.csv")
test = pd.read_csv("data/track_k_ecommerce_returns_test.csv")

In [3]:
from sklearn.tree import DecisionTreeRegressor

X_threshold = train['price_eur']
y = train['return_rate_pct']

threshold_finder = DecisionTreeRegressor(max_depth=1, random_state=42)
threshold_finder.fit(X_threshold.values.reshape(-1, 1), y)

optimal_threshold = threshold_finder.tree_.threshold[0]
print(f"The optimal split point is: {optimal_threshold:.2f}")

The optimal split point is: 173.96


We're using DecisionTreeRegressor at 1 depth to make the sole most efficient cut of our data

In [4]:
df_low_price = train[train['price_eur'] <= optimal_threshold]
df_high_price = train[train['price_eur'] > optimal_threshold]

X_low = df_low_price.drop(columns=['return_rate_pct'])
y_low = df_low_price['return_rate_pct']

X_high = df_high_price.drop(columns=['return_rate_pct'])
y_high = df_high_price['return_rate_pct']

Separating the datas into two frames so we can train two models.

In [5]:
#Test on RdForest

model_low_tier = RandomForestRegressor(n_estimators=100, random_state=42)
model_high_tier = RandomForestRegressor(n_estimators=100, random_state=42)

model_low_tier.fit(X_low, y_low)
print("Low_Done")
model_high_tier.fit(X_high, y_high)
print("High_Done")

Low_Done
High_Done


In [8]:
def predict_return_rate(new_data_df, threshold, model_low, model_high):
    #To hold onto the predictions
    predictions = np.zeros(len(new_data_df))
    
    #Sort of boolean list that serve as a filter
    is_low = new_data_df['price_eur'] <= threshold #[True False True ...] True for low tier, False for high tier
    is_high = new_data_df['price_eur'] > threshold #[True False True ...] True for high tier, False for low tier
    
    #Low tier predictions
    if is_low.any():
        #New list with index of the low tiers + their features
        X_new_low = new_data_df.loc[is_low, model_low.feature_names_in_]

        #Predictions[is_low] targets the index of the low tiers from above
            # [True False True ...] => Index 1 and 3
        predictions[is_low] = model_low.predict(X_new_low)
        
    # High tier predictions
    if is_high.any():
        #Same principle as low tier, but for high tier

        X_new_high = new_data_df.loc[is_high, model_high.feature_names_in_]
        predictions[is_high] = model_high.predict(X_new_high)
    
        
    return predictions

final_predictions = predict_return_rate(test, optimal_threshold, model_low_tier, model_high_tier)

#print mse
final_mse = mean_squared_error(test['return_rate_pct'], final_predictions)
print(f"Final MSE on test set: {final_mse:.4f}")


Final MSE on test set: 25.4354


We have a better MSE than previously using RdForest(28.57)

In [ ]:
# Here we are going to compare Ridge, Lasso and ElasticNet, each with its own set of hyperparameters. 
# We will use nested cross-validation to do that, which allows us to get an unbiased estimate of the test performance of the best model.

from torch import threshold
from xgboost import XGBRegressor as XGBoost


# Gradient Boost
from sklearn.ensemble import GradientBoostingRegressor

#Adaboost
from sklearn.ensemble import AdaBoostRegressor

#SVM
from sklearn import svm


outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)



param_ridge = {"alpha": np.logspace(-2, 4, 20)}
param_lasso = {"alpha": np.logspace(-4, 1, 20)}
param_elasticnet = {"alpha": np.logspace(-4, 1, 20), "l1_ratio": np.linspace(0, 1, 20)}
param_randomforest = {"n_estimators": [50, 100], "max_depth": [None, 5, 10]}


#NOTE: AI GENERATED PARAMS
param_xgboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2]
}

param_gradientboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7], 
    "learning_rate": [0.01, 0.1, 0.2]
}

param_adaboost = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1, 0.2]
}

param_svm = {
    "C": [0.1, 1, 10],
    "kernel": ['linear', 'rbf']
}


models = {
    'ridge': Ridge(),
    'lasso': Lasso(),
    'elasticnet': ElasticNet(),
    'randomforest': RandomForestRegressor(random_state=42),
    'xgboost': XGBoost(),
    'gradientboost': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'svm': svm.SVR()
}

param_grids = {
    'ridge': param_ridge,
    'lasso': param_lasso,
    'elasticnet': param_elasticnet,
    'randomforest': param_randomforest,
    'xgboost': param_xgboost,
    'gradientboost': param_gradientboost,
    'adaboost': param_adaboost,
    'svm': param_svm
}



outer_scores = {name: [] for name in models.keys()}


optimal_threshold = threshold_finder.tree_.threshold[0]

X = train.drop(columns=['return_rate_pct'])
y = train['return_rate_pct']

for train_idx, val_idx in outer_cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    X_train_low, y_train_low = X_train[train_low_mask], y_train[train_low_mask]
    X_train_high, y_train_high = X_train[~train_low_mask], y_train[~train_low_mask]
    
    X_val_low, y_val_low = X_val[val_low_mask], y_val[val_low_mask]
    X_val_high, y_val_high = X_val[~val_low_mask], y_val[~val_low_mask]

    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    for name, model in models.items():
        # Tune on Low segment
        grid_low = GridSearchCV(model, param_grids[name], cv=inner_cv)
        grid_low.fit(X_train_low, y_train_low)
        
        # Tune on High segment
        grid_high = GridSearchCV(model, param_grids[name], cv=inner_cv)
        grid_high.fit(X_train_high, y_train_high)
        
        preds = np.zeros(len(X_val))
        
        # Predict using tuned models
        if len(X_val_low) > 0:
            preds[val_low_mask] = grid_low.best_estimator_.predict(X_val_low)
        if len(X_val_high) > 0:
            preds[~val_low_mask] = grid_high.best_estimator_.predict(X_val_high)
            
        fold_mse = mean_squared_error(y_val, preds)
        outer_scores[name].append(fold_mse)

for name, scores in outer_scores.items():
    print(f"{name} | Avg MSE: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")



ridge | Avg MSE: 57.1191 | Std: 1.8456
lasso | Avg MSE: 57.1578 | Std: 1.6153
elasticnet | Avg MSE: 57.3180 | Std: 1.9145
randomforest | Avg MSE: 28.3466 | Std: 1.8956
xgboost | Avg MSE: 26.1514 | Std: 1.7260
gradientboost | Avg MSE: 26.3188 | Std: 1.9818
adaboost | Avg MSE: 32.1272 | Std: 2.6520
svm | Avg MSE: 36.1346 | Std: 1.7042


In [33]:
def print_mse(predictions_list, true_values):
    for name, predictions in predictions_list.items():
        mse = mean_squared_error(true_values, predictions)
        print(f"{name} MSE: {mse:.4f}")


In [32]:
test_predictions = {name: np.zeros(len(test)) for name in models.keys()}
is_low_test = test['price_eur'] <= optimal_threshold
is_high_test = test['price_eur'] > optimal_threshold
features = train.drop(columns=['return_rate_pct']).columns


for name, clf in models.items():
    if is_low_test.any():
        grid_low = GridSearchCV(clf, globals()[f"param_{name}"], cv=KFold(n_splits=3, shuffle=True, random_state=42))
        grid_low.fit(X_low, y_low) 
        
        test_predictions[name][is_low_test] = grid_low.predict(test.loc[is_low_test, features])
        
    if is_high_test.any():
        grid_high = GridSearchCV(clf, globals()[f"param_{name}"], cv=KFold(n_splits=3, shuffle=True, random_state=42))
        grid_high.fit(X_high, y_high) 
        
        test_predictions[name][is_high_test] = grid_high.predict(test.loc[is_high_test, features])


print_mse(test_predictions, test['return_rate_pct'])

ridge MSE: 48.7461
lasso MSE: 48.4717
elasticnet MSE: 48.6359
randomforest MSE: 24.6918
xgboost MSE: 23.7512
gradientboost MSE: 23.0355
adaboost MSE: 26.9781
svm MSE: 29.9661


KFold(n_splits=3, random_state=42, shuffle=True)
